## Neuroprobe Quickstart

This notebook provides a quick overview of how to use the Neuroprobe library to load data and evaluate models. Once you have the library installed and the data downloaded, you can run through this notebook to get a sense of how to work with the data and get the train/test splits.

Once you've run through this notebook, check out the [logistic regression runner](https://github.com/insight-neuro/neuroprobe/blob/main/examples/logistic_regression_runner.py) for an example of how to set up a full training and evaluation pipeline using the Runner class.

In [ ]:
import os

os.environ["ROOT_DIR_BRAINTREEBANK"] = # TODO: Set this to the path where the BrainTreebank data is stored

The `NeuroprobeConfig` class provides the configuration for a Neuroprobe evaluation, including the data directory, sampling rate, and other parameters. You can create a config object and use it to access the data and set up your evaluation pipeline. Check out [config.py](https://github.com/insight-neuro/neuroprobe-runner/blob/main/neuroprobe/config.py) for more details on the individual fields.

Note that the configuration object can't be changed after it's created, so pass any overrides as arguments when you create the config object. For example, to set the data directory to a custom path, you can do:

```python
config = NeuroprobeConfig(data_dir="/path/to/data")
```

Otherwise, it'll look for the data directory in the `ROOT_DIR_BRAINTREEBANK` environment variable, as below.

In [15]:
from neuroprobe import NeuroprobeConfig

config = NeuroprobeConfig()

print("Expected braintreebank data at:", config.data_dir)
print("Sampling rate:", config.sampling_rate, "Hz")

Expected braintreebank data at: /cluster/scratch/bradya/neuroprobe/wang_barbu_braintreebank_2023
Sampling rate: 2048 Hz


## The BrainTreebank Subject

A subject in the BrainTreebank dataset can be loaded using the `BrainTreebankSubject` class, providing an interface for accessing the neural data of any given trial.

In [3]:
from neuroprobe import BrainTreebankSubject

subject = BrainTreebankSubject(config, subject_id=1, keep_files_open=True)
print("Loaded subject", subject.subject_id)
print("Available trials:", subject.trials)
print("Start and end of trial 1:", subject.trial_interval(1))

Loaded subject 1
Available trials: [0, 1, 2]
Start and end of trial 1: (0.0, 10449.7109375)


Now, to load the data for a specific trial, you can use the `load_neural_data` method. If no start and end times are provided, it will load the entire trial (this can be very large, so be careful!).

The loaded data will be returned as a `CraneData` object, which contains the neural signals, channel labels, and channel coordinates, as well as other trial metadata.

In [4]:
data = subject.load_neural_data(trial_id=1, start=0.0, end=50.0)  # Load the first 50 seconds of trial 1

print("Neural Data Shape:", data.signals.shape)
print("Channel Ids (First 10):", data.channel_labels[:10])
print("Channel Coordinates (First 10):", data.channel_coordinates[:10])

Neural Data Shape: torch.Size([120, 102400])
Channel Ids (First 10): ['T1bIc1', 'T1bIc2', 'T1bIc3', 'T1bIc4', 'T1bIc5', 'T1bIc6', 'T1bIc7', 'T1bIc8', 'T1cIf10', 'T1cIf11']
Channel Coordinates (First 10): tensor([[ -86., -115., -125.],
        [ -84., -117., -125.],
        [ -81., -119., -125.],
        [ -78., -121., -125.],
        [ -75., -122., -125.],
        [ -72., -124., -126.],
        [ -69., -125., -126.],
        [ -67., -129., -125.],
        [ -81., -144., -120.],
        [ -79., -144., -120.]])


## BrainTreebank Datasets

The `BrainTreebankDataset` class provides an interface for accessing the neural data for a specific trial/task pair, as well as computes the associated labels for the specified task.

The Dataset's output is a `Neuroprobeta` object, which contains the same neural data and metadata as the `CraneData` object returned by the Subject, but also includes a `labels` attribute containing the labels for the specified task.

In [ ]:
from neuroprobe import BrainTreebankDataset

dataset = BrainTreebankDataset(config, subject, trial_id=1, task="volume")

print("Items in the dataset:", len(dataset))
print("First item's label:", dataset[0].label)

# Dataset's signals and labels are convenience functions that stack the 
# signals and labels from all items in the dataset into a single NumPy array
print("Full signals shape:", dataset.signals.shape)  # (num_samples, num_channels, num_timepoints)
print("Full labels shape:", dataset.labels.shape)

Items in the dataset: 3500
First item's label: 1
Full signals shape: (3500, 120, 2048)
Full labels shape: (3500,)


## Train / Test Splits

Neuroprobe provides 3 types of train/test splits: within-session, cross-session, and cross-subject. However, we recommend you use the `Runner` class to handle the train/test splits and evaluation pipeline for you, as it simplifies the process and ensures that everything is set up correctly. 

Below is an example of how to get the within-session splits for a specific trial and task. To save memory, the splits are returned as a generator, so you can iterate through them one at a time. Each fold is a dictionary containing datasets for the train, validation, and test sets.

In [16]:
from neuroprobe import splits

# Also available: cross_session_splits and cross_subject_splits
folds = splits.within_session_splits(config, subject, test_trial_id=1, task="volume")

folds = list(folds)  # Convert generator to list to see all folds at once
print("k_folds =", len(folds))
folds

k_folds = 2


[{'train_dataset': <neuroprobe.splits._SubsetBrainTreebank at 0x14b5a6614e90>,
  'val_dataset': <neuroprobe.splits._SubsetBrainTreebank at 0x14b5a613bfb0>,
  'test_dataset': <neuroprobe.splits._SubsetBrainTreebank at 0x14b5a6d594f0>},
 {'train_dataset': <neuroprobe.splits._SubsetBrainTreebank at 0x14b5a6615b50>,
  'val_dataset': <neuroprobe.splits._SubsetBrainTreebank at 0x14b5a6d5b110>,
  'test_dataset': <neuroprobe.splits._SubsetBrainTreebank at 0x14b5a6d5a4b0>}]

## Training and Evaluating Models

Below is an example of how to train and evaluate a simple logistic regression model on the within-session splits for a specific trial and task. 

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


for fold_idx, fold in enumerate(folds):
    clf = LogisticRegression()

    train_ds = fold["train_dataset"]
    test_ds = fold["test_dataset"]

    X_train = train_ds.signals  # shape: (num_samples, num_channels, num_timepoints)
    y_train = train_ds.labels  # shape: (num_samples,)

    # Flatten the features to shape: (num_samples, num_channels * num_timepoints)
    X_train = X_train.reshape(X_train.shape[0], -1)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)

    clf.fit(X_train, y_train)

    X_test = test_ds.signals.reshape(test_ds.signals.shape[0], -1)
    X_test = scaler.transform(X_test)
    y_test = test_ds.labels

    # Evaluate model
    train_score = clf.score(X_train, y_train)
    test_score = clf.score(X_test, y_test)
    print(f"Fold {fold_idx}: Train accuracy: {train_score:.3f} | Test accuracy: {test_score:.3f}")

Fold 0: Train accuracy: 0.997 | Test accuracy: 0.527
Fold 1: Train accuracy: 0.995 | Test accuracy: 0.573


Now, check out the [logistic regression runner](https://github.com/insight-neuro/neuroprobe/blob/main/examples/logistic_regression_runner.py) for an example of how to set up a full training and evaluation pipeline using the `NeuroprobeRunner` class.